# razorpdf quickstart

Hybrid local-first PDF parsing: deterministic Rust geometry for 90%+ of pages, selective sub-page VLM for the rest.
Outputs clean Markdown plus grounded RAG chunks with bounding boxes.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shridhargsabat/razorpdf/blob/main/docs/notebook/razorpdf_quickstart.ipynb)

In [ ]:
%pip install -q razorpdf pymupdf matplotlib

In [ ]:
import urllib.request

URL = "https://arxiv.org/pdf/1706.03762.pdf"  # Attention Is All You Need
PDF = "/tmp/attention.pdf"
urllib.request.urlretrieve(URL, PDF)
print("downloaded", PDF)

In [ ]:
import razorpdf

doc = razorpdf.parse_file(PDF)
print(f"{doc['total_pages']} pages, {len(doc['chunks'])} chunks, "
      f"{doc['vlm_fallback_regions']} VLM regions, {doc['elapsed_ms']:.0f} ms")
print(doc["markdown"][:600])

In [ ]:
# Chunk inventory: id, page, bbox, heading trail, tokens, VLM provenance
for c in doc["chunks"][:8]:
    print(c["chunk_id"], "p%d" % c["page_index"],
          [round(v) for v in c["bbox"].values()],
          " > ".join(c["breadcrumb"]),
          c["token_estimate"], "VLM" if c["parsed_by_vlm"] else "geo")

In [ ]:
# Visualize Table 1's chunk bboxes on page 6 (index 5)
import pymupdf
from PIL import Image, ImageDraw

PAGE = 5
page = pymupdf.open(PDF)[PAGE]
pix = page.get_pixmap(matrix=pymupdf.Matrix(2, 2))
img = Image.frombytes("RGB", (pix.width, pix.height), pix.samples)
s = pix.width / page.rect.width
draw = ImageDraw.Draw(img)
for c in doc["chunks"]:
    if c["page_index"] != PAGE:
        continue
    b = c["bbox"]
    box = [b["x0"] * s, b["y0"] * s, b["x1"] * s, b["y1"] * s]
    table = c["markdown"].lstrip().startswith("|")
    draw.rectangle(box, outline=(22, 163, 74) if table else (37, 99, 235),
                   width=4 if table else 2)
img.save("/tmp/attention_p6_boxes.png")
img.resize((img.width // 2, img.height // 2))